In [4]:
# --- Django Standalone-Setup (setzt das Projekt aus Lab 1 voraus) ---
import os
import sys
import django


PROJEKT_DIR = os.path.abspath(".")
if PROJEKT_DIR not in sys.path:
    sys.path.insert(0, PROJEKT_DIR)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "locallibrary.settings")


os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()  

from catalog.models import Author, Genre, Language, Book, BookInstance
print("django.setup() OK — Django", django.get_version())

ModuleNotFoundError: No module named 'django'

In [ ]:
# Hilfsdaten anlegen — get_or_create() gibt (objekt, created_bool) zurück.
from datetime import date

# Genres
sf, _   = Genre.objects.get_or_create(name="Science-Fiction")
fant, _ = Genre.objects.get_or_create(name="Fantasy")
roman, _ = Genre.objects.get_or_create(name="Roman")

# Sprachen
de, _ = Language.objects.get_or_create(name="Deutsch")
en, _ = Language.objects.get_or_create(name="Englisch")

# Autor:innen
tolkien, _ = Author.objects.get_or_create(
    first_name="J. R. R.", last_name="Tolkien",
    defaults={"date_of_birth": date(1892, 1, 3)},
)
lem, _ = Author.objects.get_or_create(
    first_name="Stanisław", last_name="Lem",
    defaults={"date_of_birth": date(1921, 9, 12)},
)
le_guin, _ = Author.objects.get_or_create(
    first_name="Ursula K.", last_name="Le Guin",
    defaults={"date_of_birth": date(1929, 10, 21)},
)

# Bücher (ISBN ist unique -> idempotenter Match-Schlüssel)
hobbit, _ = Book.objects.get_or_create(
    isbn="9780261103283",
    defaults={"title": "Der Hobbit", "author": tolkien,
              "language": de, "summary": "Bilbo Beutlin auf Abenteuerreise."},
)
solaris, _ = Book.objects.get_or_create(
    isbn="9780156027601",
    defaults={"title": "Solaris", "author": lem,
              "language": de, "summary": "Erstkontakt mit einem Ozean-Planeten."},
)
darkness, _ = Book.objects.get_or_create(
    isbn="9780441478125",
    defaults={"title": "The Left Hand of Darkness", "author": le_guin,
              "language": en, "summary": "Geschlecht und Gesellschaft auf Gethen."},
)

# M2M-Genres zuweisen (set() ist idempotent)
hobbit.genre.set([fant])
solaris.genre.set([sf, roman])
darkness.genre.set([sf])

# BookInstances (Exemplare). UUID-PK -> wir matchen über (book, imprint).
BookInstance.objects.get_or_create(
    book=hobbit,
    defaults={"status": "a"},
)
BookInstance.objects.get_or_create(
    book=solaris, 
    defaults={"status": "o", "due_back": date(2026, 6, 15)},
)
BookInstance.objects.get_or_create(
    book=darkness, 
    defaults={"status": "a"},
)

print("Bücher:", Book.objects.count(),
      "| Autor:innen:", Author.objects.count(),
      "| Exemplare:", BookInstance.objects.count())
# => Bücher: 3 | Autor:innen: 3 | Exemplare: 3